# Silver project

## Purpose

Create the canonical project-level Silver table used by future project-region matching, flood-control reconciliation, portfolio analytics, delivery analytics, and Gold project dimensions and facts.

This table consolidates component-level evidence while preserving conflicts and data-quality findings.

One output row represents one canonical project for the selected source snapshot and Silver processing rule set.

# 1. Read component-level project evidence

The project table is built entirely from `02-silver`.silver_dpwh_project_component.

Bronze data is not re-read and re-cleaned.

This ensures a single source of truth for parsing, standardization, quality controls, lineage, and mapping results.

Expected validation:

- All project inputs originate from the component table
- Source lineage remains intact
- Project accounting reconciles to component accounting
``

In [ ]:
USE CATALOG `buildabida-capstone`;

CREATE OR REPLACE TEMPORARY VIEW project_component_input AS
SELECT
    *
FROM `02-silver`.silver_dpwh_project_component;

# 2. Resolve project-level attributes conservatively

Project-level values are only published when component-level evidence resolves consistently.

When exactly one meaningful value exists, the value becomes the canonical project value.

When no meaningful value exists, the value remains unresolved.

When multiple conflicting values exist, the conflict remains visible rather than being resolved automatically.

Expected validation:

- Conflicts are reported explicitly
- No arbitrary source row is selected
- Attribute resolution remains deterministic


# 3. Protect against budget double counting

Repeated component rows must not automatically increase project investment values.

The project budget represents the reported project or contract budget and does not imply actual payments or disbursements.

Repeated identical values may resolve to a single canonical budget value.

Multiple conflicting budget values remain unresolved until a governed business rule is approved.

Expected validation:

- Repeated values are not blindly summed
- Conflicting values remain visible
- Budget resolution status is reported
- Project-level investment remains explainable
``

# 4. Publish one project-level record

The output grain is one canonical project per selected source snapshot and Silver processing rule set.

The table preserves:

- Project-level attributes
- Conflict indicators
- Delivery-rule fields
- Mapping results
- Full source lineage
- Deterministic processing identity

Expected validation:

- One row per project grain
- No duplicate project identities
- Component accounting reconciles to project accounting
- Project outputs remain traceable back to source evidence

# Summary

Created `02-silver`.silver_project at one canonical project per selected source snapshot and Silver run.

Component-level evidence remains available in `02-silver`.silver_dpwh_project_component. Conflicting project attributes remain visible rather than being automatically resolved. Budget protection prevents unsupported double counting of repeated source records.

Requires Databricks execution.

Next dependency: `02-silver`.silver_flood_control_component.